# Этап B: выдачи всех поисковых конфигураций

Запускается после того, как набор запросов собран целиком: синтетические
с этапа A плюс пятьдесят написанных руками, с проверенной разметкой.

Подключить **три** входа: датасет `ru-fin-retrieval`, датасет
`ru-fin-queries` и вывод ноутбука этапа A (вкладка **Your Work**).
меню **Settings** → **Accelerator** → **GPU T4 ×2**; там же проверить,
что интернет включён (пункт читается как **Turn off internet**).


In [ ]:
!pip install -q -U sentence-transformers


In [ ]:
import glob, os, shutil, zipfile

# Kaggle монтирует датасеты по-разному: то прямо в /kaggle/input/<имя>,
# то глубже, внутри /kaggle/input/datasets/<владелец>/<имя>. Поэтому путь
# не задаётся, а источник ищется по содержимому: нужен каталог, где лежит
# пакет rufin или корпус. Глубина ограничена, чтобы не обходить всё подряд.
def find_source(root='/kaggle/input', max_depth=4):
    archives = []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs or any(f.startswith('acts.jsonl') for f in files):
            return cur, None
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


def show_tree(root='/kaggle/input', max_depth=3):
    for cur, dirs, files in os.walk(root):
        depth = cur.count('/') - root.count('/')
        if depth >= max_depth:
            dirs.clear()
        print('   ' * depth + os.path.basename(cur) + '/')
        for f in sorted(files)[:6]:
            print('   ' * (depth + 1) + f)


src, archive = find_source()
if src is None:
    print('что ноутбук видит на входе:')
    show_tree()
    raise SystemExit('датасет не подключён: панель справа -> + Add Input -> ru-fin-retrieval -> +')

print('источник:', src)
if archive:
    print('лежит архивом, распаковываю:', archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for item in glob.glob(os.path.join(src, '*')):
        dst = os.path.join('/kaggle/working', os.path.basename(item))
        if os.path.isdir(item):
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)

%cd /kaggle/working
print('\nв рабочей папке:', sorted(os.listdir('.')))

assert os.path.isdir('rufin'), f'пакета rufin нет; в источнике было: {sorted(os.listdir(src))}'
# Kaggle иногда разжимает архив при создании датасета, имя корпуса плавает
assert glob.glob('acts.jsonl*'), 'корпуса нет'
print('всё на месте:', sorted(glob.glob('acts.jsonl*')))


## Где лежат эмбеддинги этапа A

Путь зависит от того, как назван ноутбук этапа A, поэтому он ищется,
а не вписывается руками.


In [ ]:
import glob
found = glob.glob('/kaggle/input/*/embeddings')
print('найдено:', found)
EMB = found[0] if found else None
print('используем:', EMB)
assert EMB, 'вывод этапа A не подключён — добавьте его через + Add Input'


## Прогон

Около полутора часов; больше всего времени уходит на переранжирование
(двести запросов по пятьдесят кандидатов).


In [ ]:
!cd /kaggle/working && python run_phase_b.py \
    --queries /kaggle/input/ru-fin-queries/queries.jsonl \
    --embeddings {EMB} \
    --out /kaggle/working/runs


## Что получилось


In [ ]:
import json, os
r = json.load(open('/kaggle/working/runs/report_phase_b.json'))
print('запросов:', r['queries'])
for x in r['runs']:
    print(f"   {x['config']:<12} {x['retrieval']}")
size = sum(os.path.getsize(os.path.join('/kaggle/working/runs', f))
           for f in os.listdir('/kaggle/working/runs'))
print(f'\nвсего {size/1048576:.1f} МБ — скачать папку runs целиком')


Скачать папку `runs` и положить в проект как `data/runs`.
Дальше на маке: `make metrics`, `make errors`, `make latency`.
